In [29]:
import itertools
import time
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from numpy.linalg import LinAlgError as NpLinAlgError
from scipy.linalg import LinAlgError as SpLinAlgError
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from torch import nn, optim
from torch.utils.data import ConcatDataset, DataLoader, TensorDataset

warnings.filterwarnings("ignore")

# Настройки для красивых графиков
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")


DATA_DIR = "./Data/"
MODELS_DIR = "./models/"

# Тикеры для обучения
CAR_TICKERS = ["Volkswagen"]
# ["Volkswagen", "Renault", "Li Auto", "NIO"]
SNEAKERS_TICKERS = ["Nike"]
# ["Nike", "Puma"]

In [43]:
car_data = pd.read_csv(
    DATA_DIR + "car_data_preprocessed.csv", parse_dates=["Date"], index_col="Date"
)
sneakers_data = pd.read_csv(
    DATA_DIR + "sneakers_data_preprocessed.csv", parse_dates=["Date"], index_col="Date"
)

car_data.dropna(inplace=True)
sneakers_data.dropna(inplace=True)

# Сплит + нормализация

In [44]:
SPLIT_RATIOS = (0.7, 0.15, 0.15)  # (train, val, test)

# Колонки, которые НЕ нужно нормализовать (бинарные флаги уже в [0, 1])
BINARY_COLS = [
    "is_month_start",
    "is_month_end",
    "is_quarter_start",
    "is_quarter_end",
    "is_year_start",
    "is_year_end",
    "is_friday",
    "is_monday",
]


# TODO прикрутить кроссвалидацию,а не просто статический сплит
def chronological_split(df, ratios):
    """
    Разбивает датафрейм хронологически на train/val/test.

    Параметры:
        df: датафрейм с DatetimeIndex
        ratios: кортеж (train_ratio, val_ratio, test_ratio)

    Возвращает:
        (train_df, val_df, test_df)
    """
    assert sum(ratios) == 1.0, f"Сумма ratios должна быть 1.0, получено {sum(ratios)}"

    n = len(df)
    train_end = int(n * ratios[0])
    val_end = train_end + int(n * ratios[1])

    train = df.iloc[:train_end]
    val = df.iloc[train_end:val_end]
    test = df.iloc[val_end:]

    return train, val, test


# TODO подумать, хороший ли вариант нормализовать именно минмаксом, возможно стоит взять какой-то другой scaler
def normalize_datasets(train, val, test, binary_cols):
    """
    Fit MinMaxScaler на train, transform на train/val/test.
    Бинарные колонки не нормализуются.

    Возвращает:
        (train_norm, val_norm, test_norm, scaler, cols_to_scale)
    """
    # Определяем колонки для нормализации (все числовые, кроме бинарных)
    all_numeric = train.select_dtypes(include=[np.number]).columns.tolist()
    cols_to_scale = [c for c in all_numeric if c not in binary_cols]
    cols_to_keep = [c for c in train.columns if c not in cols_to_scale]

    print(f"      Колонок для нормализации: {len(cols_to_scale)}")
    print(f"      Колонок без изменений: {len(cols_to_keep)}")

    # Fit на TRAIN только!
    scaler = MinMaxScaler()
    scaler.fit(train[cols_to_scale])

    # Transform всех трёх частей
    train_scaled = pd.DataFrame(
        scaler.transform(train[cols_to_scale]), columns=cols_to_scale, index=train.index
    )
    val_scaled = pd.DataFrame(
        scaler.transform(val[cols_to_scale]), columns=cols_to_scale, index=val.index
    )
    test_scaled = pd.DataFrame(
        scaler.transform(test[cols_to_scale]), columns=cols_to_scale, index=test.index
    )

    # Добавляем обратно ненормализованные колонки
    if cols_to_keep:
        train_scaled = pd.concat([train_scaled, train[cols_to_keep]], axis=1)
        val_scaled = pd.concat([val_scaled, val[cols_to_keep]], axis=1)
        test_scaled = pd.concat([test_scaled, test[cols_to_keep]], axis=1)

    # Восстанавливаем порядок колонок как в оригинале
    train_scaled = train_scaled[train.columns]
    val_scaled = val_scaled[train.columns]
    test_scaled = test_scaled[train.columns]

    return train_scaled, val_scaled, test_scaled, scaler, cols_to_scale


In [45]:
car_train, car_val, car_test = chronological_split(car_data, SPLIT_RATIOS)
car_train_norm, car_val_norm, car_test_norm, car_scaler, car_scaled_cols = (
    normalize_datasets(car_train, car_val, car_test, BINARY_COLS)
)

sneakers_train, sneakers_val, sneakers_test = chronological_split(
    sneakers_data, SPLIT_RATIOS
)
(
    sneakers_train_norm,
    sneakers_val_norm,
    sneakers_test_norm,
    sneakers_scaler,
    sneakers_scaled_cols,
) = normalize_datasets(sneakers_train, sneakers_val, sneakers_test, BINARY_COLS)

      Колонок для нормализации: 118
      Колонок без изменений: 8
      Колонок для нормализации: 62
      Колонок без изменений: 8


In [46]:
# Scaler'ы (для обратного преобразования при подсчёте метрик)
joblib.dump(car_scaler, MODELS_DIR + "car_scaler.pkl")
joblib.dump(sneakers_scaler, MODELS_DIR + "sneakers_scaler.pkl")

['./models/sneakers_scaler.pkl']

# Аримы

In [36]:
# Параметры для grid search
ARIMA_P_RANGE = range(4)  # p: 0-3
ARIMA_D_RANGE = range(3)  # d: 0-2
ARIMA_Q_RANGE = range(4)  # q: 0-3

SARIMA_P_RANGE = range(3)  # p: 0-2
SARIMA_D_RANGE = range(2)  # d: 0-1
SARIMA_Q_RANGE = range(3)  # q: 0-2
SARIMA_P_SEASONAL_RANGE = range(2)  # P: 0-1
SARIMA_D_SEASONAL_RANGE = range(2)  # D: 0-1
SARIMA_Q_SEASONAL_RANGE = range(2)  # Q: 0-1

In [37]:
# TODO сделать подбор гиперпараметров через optuna, а не в тупую грид сёрчем

ARIMA_FIT_ERRORS = (
    NpLinAlgError,  # Сингулярная матрица Гессе (самая частая)
    SpLinAlgError,  # То же самое, но из scipy (statsmodels использует обе)
    ValueError,  # Некорректная инициализация, нестационарный ряд
    RuntimeError,  # Оптимизатор не сошёлся
    OverflowError,  # Переполнение при вычислении правдоподобия
    FloatingPointError,  # проблемы с NaN/Inf в вычислениях
)


def grid_search_arima(train_series, p_range, d_range, q_range):
    """
    Grid search для ARIMA по AIC на train с обработкой исключений.
    """
    best_aic = np.inf
    best_model = None
    best_params = None

    param_grid = list(itertools.product(p_range, d_range, q_range))
    n_total = len(param_grid)
    n_success = 0

    for p, d, q in param_grid:
        try:
            model = ARIMA(train_series, order=(p, d, q))
            fitted_model = model.fit()
            n_success += 1

            if fitted_model.aic < best_aic:
                best_aic = fitted_model.aic
                best_model = fitted_model
                best_params = (p, d, q)

        except ARIMA_FIT_ERRORS as e:
            print(f"[ARIMA] Ошибка при order=({p},{d},{q}): {type(e).__name__} - {e}")
            continue

    print(f"[ARIMA] Успешно обучено: {n_success} из {n_total}")

    if best_model is None:
        return None, None, np.inf

    return best_model, best_params, best_aic


def grid_search_sarima(
    train_series, p_range, d_range, q_range, P_range, D_range, Q_range, s
):
    """
    Grid search для SARIMA по AIC на train с обработкой исключений.
    """
    best_aic = np.inf
    best_model = None
    best_params = None

    param_grid = list(
        itertools.product(p_range, d_range, q_range, P_range, D_range, Q_range)
    )
    n_total = len(param_grid)
    n_success = 0

    for p, d, q, P, D, Q in param_grid:
        try:
            model = SARIMAX(train_series, order=(p, d, q), seasonal_order=(P, D, Q, s))
            fitted_model = model.fit()
            n_success += 1

            if fitted_model.aic < best_aic:
                best_aic = fitted_model.aic
                best_model = fitted_model
                best_params = (p, d, q, P, D, Q, s)

        except ARIMA_FIT_ERRORS as e:
            print(
                f"[SARIMA] Ошибка при order=({p},{d},{q})x({P},{D},{Q})_{s}: "
                f"{type(e).__name__} - {e}"
            )
            continue

    print(f"[SARIMA] Успешно обучено: {n_success} из {n_total}")

    if best_model is None:
        return None, None, np.inf

    return best_model, best_params, best_aic

In [38]:
def train_and_evaluate_arima(
    train_series, val_series, test_series, ticker_name, model_type
):
    """
    Обучение и оценка ARIMA-модели для одного тикера.
    """
    results = {
        "ticker": ticker_name,
        "model_type": model_type,
        "train_time": np.nan,
        "inference_time": np.nan,
        "mae": np.nan,
        "rmse": np.nan,
        "mape": np.nan,
        "best_params": None,
        "best_aic": np.nan,
    }

    # Объединяем train и val, чтобы модель видела больше данных
    train_val_series = pd.concat([train_series, val_series])

    # Обучение
    start_train = time.time()

    if model_type == "arima":
        best_model, best_params, best_aic = grid_search_arima(
            train_val_series, ARIMA_P_RANGE, ARIMA_D_RANGE, ARIMA_Q_RANGE
        )
    elif model_type == "sarima_5":
        best_model, best_params, best_aic = grid_search_sarima(
            train_val_series,
            SARIMA_P_RANGE,
            SARIMA_D_RANGE,
            SARIMA_Q_RANGE,
            SARIMA_P_SEASONAL_RANGE,
            SARIMA_D_SEASONAL_RANGE,
            SARIMA_Q_SEASONAL_RANGE,
            s=5,
        )
    elif model_type == "sarima_21":
        best_model, best_params, best_aic = grid_search_sarima(
            train_val_series,
            SARIMA_P_RANGE,
            SARIMA_D_RANGE,
            SARIMA_Q_RANGE,
            SARIMA_P_SEASONAL_RANGE,
            SARIMA_D_SEASONAL_RANGE,
            SARIMA_Q_SEASONAL_RANGE,
            s=21,
        )
    else:
        raise ValueError(f"Неизвестный тип модели: {model_type}")

    end_train = time.time()
    results["train_time"] = end_train - start_train

    # Обработка случая, когда ни одна модель не обучилась
    if best_model is None:
        print(
            f"[{ticker_name} | {model_type}] "
            f"Не удалось обучить ни одну модель из сетки параметров. "
            f"Пропускаем."
        )
        return results

    results["best_params"] = best_params
    results["best_aic"] = best_aic

    # Инференс (прогноз на размер теста)
    start_inference = time.time()
    forecast = best_model.forecast(steps=len(test_series))
    end_inference = time.time()
    results["inference_time"] = end_inference - start_inference

    # Метрики
    actual = test_series.values
    predicted = forecast.values

    results["mae"] = mean_absolute_error(actual, predicted)
    results["rmse"] = np.sqrt(mean_squared_error(actual, predicted))

    # MAPE (с защитой от деления на ноль)
    mask = actual != 0
    if mask.sum() > 0:
        results["mape"] = (
            np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100
        )
    else:
        results["mape"] = np.nan

    return results

In [39]:
# Основной цикл обучения для всех тикеров
all_results = []

print("=" * 80)
print("Обучение ARIMA-моделей")
print("=" * 80)

# Автомобили
print("\n Обучение на автомобильных данных...")
for ticker in CAR_TICKERS:
    target_col = f"{ticker}_target"
    print(f"\n  Тикер: {ticker}")

    # Берём оригинальные (не нормализованные) данные для ARIMA
    train_series = car_train[target_col]
    val_series = car_val[target_col]
    test_series = car_test[target_col]

    # ARIMA
    print("    Обучение ARIMA...")
    results = train_and_evaluate_arima(
        train_series, val_series, test_series, ticker, "arima"
    )
    all_results.append(results)
    print(f"      Параметры: {results['best_params']}, AIC: {results['best_aic']:.2f}")
    print(
        f"      MAE: {results['mae']:.4f}, RMSE: {results['rmse']:.4f}, MAPE: {results['mape']:.2f}%"
    )

    # SARIMA s=5
    print("    Обучение SARIMA (s=5)...")
    results = train_and_evaluate_arima(
        train_series, val_series, test_series, ticker, "sarima_5"
    )
    all_results.append(results)
    print(f"      Параметры: {results['best_params']}, AIC: {results['best_aic']:.2f}")
    print(
        f"      MAE: {results['mae']:.4f}, RMSE: {results['rmse']:.4f}, MAPE: {results['mape']:.2f}%"
    )

    # SARIMA s=21
    print("    Обучение SARIMA (s=21)...")
    results = train_and_evaluate_arima(
        train_series, val_series, test_series, ticker, "sarima_21"
    )
    all_results.append(results)
    print(f"      Параметры: {results['best_params']}, AIC: {results['best_aic']:.2f}")
    print(
        f"      MAE: {results['mae']:.4f}, RMSE: {results['rmse']:.4f}, MAPE: {results['mape']:.2f}%"
    )

# Кроссовки
print("\n\n👟 Обучение на данных кроссовок...")
for ticker in SNEAKERS_TICKERS:
    target_col = f"{ticker}_target"
    print(f"\n  Тикер: {ticker}")

    train_series = sneakers_train[target_col]
    val_series = sneakers_val[target_col]
    test_series = sneakers_test[target_col]

    # ARIMA
    print("    Обучение ARIMA...")
    results = train_and_evaluate_arima(
        train_series, val_series, test_series, ticker, "arima"
    )
    all_results.append(results)
    print(f"      Параметры: {results['best_params']}, AIC: {results['best_aic']:.2f}")
    print(
        f"      MAE: {results['mae']:.4f}, RMSE: {results['rmse']:.4f}, MAPE: {results['mape']:.2f}%"
    )

    # SARIMA s=5
    print("    Обучение SARIMA (s=5)...")
    results = train_and_evaluate_arima(
        train_series, val_series, test_series, ticker, "sarima_5"
    )
    all_results.append(results)
    print(f"      Параметры: {results['best_params']}, AIC: {results['best_aic']:.2f}")
    print(
        f"      MAE: {results['mae']:.4f}, RMSE: {results['rmse']:.4f}, MAPE: {results['mape']:.2f}%"
    )

    # SARIMA s=21
    print("    Обучение SARIMA (s=21)...")
    results = train_and_evaluate_arima(
        train_series, val_series, test_series, ticker, "sarima_21"
    )
    all_results.append(results)
    print(f"      Параметры: {results['best_params']}, AIC: {results['best_aic']:.2f}")
    print(
        f"      MAE: {results['mae']:.4f}, RMSE: {results['rmse']:.4f}, MAPE: {results['mape']:.2f}%"
    )

# Сводная таблица результатов
results_df = pd.DataFrame(all_results)
print("\n" + "=" * 80)
print("Сводная таблица результатов ARIMA-моделей")
print("=" * 80)
print(results_df.to_string(index=False))

Обучение ARIMA-моделей

 Обучение на автомобильных данных...

  Тикер: Volkswagen
    Обучение ARIMA...


KeyboardInterrupt: 

In [ ]:
results_df.to_csv("./results/arima_results.csv", index=False)

# MLP

In [ ]:
# === Архитектура MLP ===
class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.network(x)


# === Вспомогательные функции ===
def set_seed(seed=42):
    """Фиксация random seed для воспроизводимости."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)


def get_ticker_features(ticker, df_columns):
    """
    Отбирает признаки для конкретного тикера:
    - Лаги target этого тикера
    - Rolling-статистики target этого тикера
    - Календарные признаки
    """
    features = []

    # Лаги target
    for lag in range(1, 11):
        col = f"{ticker}_target_lag{lag}"
        if col in df_columns:
            features.append(col)

    # Rolling-статистики target
    for window in [5, 10, 20, 50]:
        for stat in ["mean", "std", "min", "max"]:
            col = f"{ticker}_target_rolling_{stat}_{window}"
            if col in df_columns:
                features.append(col)

    # Календарные признаки
    calendar_cols = [
        "day_of_week",
        "month",
        "is_friday",
        "is_monday",
        "is_month_start",
        "is_month_end",
        "is_quarter_start",
        "is_quarter_end",
        "is_year_start",
        "is_year_end",
    ]
    for col in calendar_cols:
        if col in df_columns:
            features.append(col)

    return features


def prepare_data_for_mlp(train_norm, val_norm, test_norm, ticker):
    """
    Подготавливает данные для MLP: отбирает признаки, создаёт DataLoader.

    Возвращает:
        (train_loader, val_loader, test_loader, feature_cols, target_col)
    """
    target_col = f"{ticker}_target"
    feature_cols = get_ticker_features(ticker, train_norm.columns)

    # Убеждаемся, что target не в фичах
    if target_col in feature_cols:
        feature_cols.remove(target_col)

    # Извлекаем данные
    X_train = train_norm[feature_cols].values
    y_train = train_norm[target_col].values

    X_val = val_norm[feature_cols].values
    y_val = val_norm[target_col].values

    X_test = test_norm[feature_cols].values
    y_test = test_norm[target_col].values

    # Конвертируем в тензоры
    X_train_tensor = torch.FloatTensor(X_train)
    y_train_tensor = torch.FloatTensor(y_train).unsqueeze(1)

    X_val_tensor = torch.FloatTensor(X_val)
    y_val_tensor = torch.FloatTensor(y_val).unsqueeze(1)

    X_test_tensor = torch.FloatTensor(X_test)
    y_test_tensor = torch.FloatTensor(y_test).unsqueeze(1)

    # Создаём DataLoader
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

    val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

    test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    return train_loader, val_loader, test_loader, feature_cols, target_col


def train_mlp_find_best_epoch(
    model, train_loader, val_loader, epochs=100, lr=1e-3, patience=10
):
    """
    Первый проход: обучение на train + early stopping на val.
    """
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=5, factor=0.5
    )

    best_val_loss = float("inf")
    best_epoch = 0
    patience_counter = 0

    start_time = time.time()

    for epoch in range(1, epochs + 1):
        # Training
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)

        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)

                # Проверка на NaN
                if torch.isnan(loss) or torch.isinf(loss):
                    print(f"        ⚠️  Epoch {epoch}: val loss is NaN/Inf")
                    val_loss = float("inf")
                    break

                val_loss += loss.item()
        val_loss /= len(val_loader)

        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    train_time = time.time() - start_time

    return best_epoch, best_val_loss, train_time


def train_mlp_final(model, full_train_loader, target_epochs, lr=1e-3):
    """
    Второй проход: переобучение модели на train+val ровно на target_epochs эпох.
    """
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    start_time = time.time()

    for epoch in range(target_epochs):
        model.train()
        for X_batch, y_batch in full_train_loader:
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()

    train_time = time.time() - start_time
    return model, train_time


def recursive_forecast_mlp(model, history_df, test_norm, feature_cols, target_col):
    """
    Рекурсивный multi-step forecast для MLP.

    history_df: train + val (используется как начальная история для лагов/rolling)
    test_norm: тестовые данные (нужны только календарные фичи)

    На каждом шаге:
    1. Берём последние значения признаков из истории
    2. Предсказываем 1 шаг
    3. Добавляем предсказание в историю
    4. Сдвигаем лаги и rolling-статистики
    """
    model.eval()

    # Начинаем с train+val данных (последние значения для лагов)
    history = history_df.copy()

    predictions = []

    with torch.no_grad():
        for step in range(len(test_norm)):
            # Берём последние значения признаков
            last_row = history.iloc[-1:][feature_cols].values
            X = torch.FloatTensor(last_row)

            # Предсказание
            pred_scaled = model(X).item()
            predictions.append(pred_scaled)

            # Создаём новую строку для истории
            new_row = history.iloc[-1:].copy()
            new_row[target_col] = pred_scaled

            # Обновляем лаги (сдвигаем назад)
            ticker = target_col.replace("_target", "")
            for lag in range(10, 1, -1):
                lag_col = f"{ticker}_target_lag{lag}"
                prev_lag_col = f"{ticker}_target_lag{lag - 1}"
                if lag_col in new_row.columns and prev_lag_col in new_row.columns:
                    new_row[lag_col] = new_row[prev_lag_col].values[0]

            # lag1 = текущее предсказание
            lag1_col = f"{ticker}_target_lag1"
            if lag1_col in new_row.columns:
                new_row[lag1_col] = pred_scaled

            # Обновляем rolling-статистики
            for window in [5, 10, 20, 50]:
                for stat in ["mean", "std", "min", "max"]:
                    col = f"{ticker}_target_rolling_{stat}_{window}"
                    if col in new_row.columns:
                        # Берём последние window значений из history + новое предсказание
                        recent_values = list(
                            history[target_col].values[-(window - 1) :]
                        ) + [pred_scaled]
                        if stat == "mean":
                            new_row[col] = np.mean(recent_values)
                        elif stat == "std":
                            new_row[col] = np.std(recent_values)
                        elif stat == "min":
                            new_row[col] = np.min(recent_values)
                        elif stat == "max":
                            new_row[col] = np.max(recent_values)

            # Добавляем новую строку в историю
            history = pd.concat([history, new_row], ignore_index=False)

    return np.array(predictions)


def inverse_transform_target(predictions_scaled, scaler, scaled_cols, target_col):
    """Обратное преобразование предсказаний в исходный масштаб."""
    target_idx = scaled_cols.index(target_col)

    # Создаём dummy array с правильным количеством колонок
    dummy = np.zeros((len(predictions_scaled), len(scaled_cols)))
    dummy[:, target_idx] = predictions_scaled

    # Inverse transform
    predictions_original = scaler.inverse_transform(dummy)[:, target_idx]

    return predictions_original


def evaluate_mlp(
    model, history_df, test_norm, feature_cols, target_col, scaler, scaled_cols
):
    """
    Оценка MLP: рекурсивный прогноз + метрики + тайминг.
    """
    # Рекурсивный прогноз
    start_inference = time.time()
    predictions_scaled = recursive_forecast_mlp(
        model, history_df, test_norm, feature_cols, target_col
    )
    inference_time = time.time() - start_inference

    # Inverse transform предсказаний
    predictions_original = inverse_transform_target(
        predictions_scaled, scaler, scaled_cols, target_col
    )

    # Inverse transform actual значений
    actual_scaled = test_norm[target_col].values
    actual_original = inverse_transform_target(
        actual_scaled, scaler, scaled_cols, target_col
    )

    # Метрики
    mae = mean_absolute_error(actual_original, predictions_original)
    rmse = np.sqrt(mean_squared_error(actual_original, predictions_original))
    mask = actual_original != 0
    mape = (
        np.mean(
            np.abs(
                (actual_original[mask] - predictions_original[mask])
                / actual_original[mask]
            )
        )
        * 100
        if mask.any()
        else np.nan
    )

    return predictions_original, inference_time, mae, rmse, mape


In [50]:
# === Основной цикл обучения ===
set_seed(42)

all_mlp_results = []


# Автомобили
print(" Автомобили")
for ticker in CAR_TICKERS:
    print(f" {ticker}")

    # Подготовка данных
    train_loader, val_loader, test_loader, feature_cols, target_col = (
        prepare_data_for_mlp(car_train_norm, car_val_norm, car_test_norm, ticker)
    )

    print(f"    Признаков: {len(feature_cols)}")

    # === ПЕРВЫЙ ПРОХОД: определяем best_epoch ===
    model = MLP(input_dim=len(feature_cols))
    print("    Первый проход (определение best_epoch)...")
    best_epoch, best_val_loss, _ = train_mlp_find_best_epoch(
        model, train_loader, val_loader, lr=0.0001
    )
    print(f"    Best epoch: {best_epoch}, val loss: {best_val_loss:.6f}")

    # === ОБЪЕДИНЯЕМ train + val ===
    full_train_dataset = ConcatDataset([train_loader.dataset, val_loader.dataset])
    full_train_loader = DataLoader(full_train_dataset, batch_size=32, shuffle=True)

    # === ВТОРОЙ ПРОХОД: переобучение на train+val ===
    model = MLP(input_dim=len(feature_cols))  # Пересоздаём с fresh weights
    print(f"    Второй проход (обучение на train+val, {best_epoch} эпох)...")
    model, train_time = train_mlp_final(
        model, full_train_loader, target_epochs=best_epoch
    )
    print(f"    Train time: {train_time:.2f}s")

    # === ОЦЕНКА на test ===
    # Используем train+val как историю для рекурсивного прогноза
    history_df = pd.concat([car_train_norm, car_val_norm])
    predictions, inference_time, mae, rmse, mape = evaluate_mlp(
        model,
        history_df,
        car_test_norm,
        feature_cols,
        target_col,
        car_scaler,
        car_scaled_cols,
    )

    print(
        f"    MAE: {mae:.4f}, RMSE: {rmse:.4f}, MAPE: {mape:.2f}%, Inference: {inference_time:.4f}s"
    )

    all_mlp_results.append(
        {
            "ticker": ticker,
            "model": "MLP",
            "best_epoch": best_epoch,
            "train_time": train_time,
            "inference_time": inference_time,
            "mae": mae,
            "rmse": rmse,
            "mape": mape,
            "best_val_loss": best_val_loss,
        }
    )

# Кроссовки
print(" Кроссовки")
for ticker in SNEAKERS_TICKERS:
    print(f" {ticker}")

    # Подготовка данных
    train_loader, val_loader, test_loader, feature_cols, target_col = (
        prepare_data_for_mlp(
            sneakers_train_norm, sneakers_val_norm, sneakers_test_norm, ticker
        )
    )

    print(f"    Признаков: {len(feature_cols)}")

    # === ПЕРВЫЙ ПРОХОД: определяем best_epoch ===
    model = MLP(input_dim=len(feature_cols))
    print("    Первый проход (определение best_epoch)...")
    best_epoch, best_val_loss, _ = train_mlp_find_best_epoch(
        model, train_loader, val_loader
    )
    print(f"    Best epoch: {best_epoch}, val loss: {best_val_loss:.6f}")

    # === ОБЪЕДИНЯЕМ train + val ===
    full_train_dataset = ConcatDataset([train_loader.dataset, val_loader.dataset])
    full_train_loader = DataLoader(full_train_dataset, batch_size=32, shuffle=True)

    # === ВТОРОЙ ПРОХОД: переобучение на train+val ===
    model = MLP(input_dim=len(feature_cols))  # Пересоздаём с fresh weights
    print(f"    Второй проход (обучение на train+val, {best_epoch} эпох)...")
    model, train_time = train_mlp_final(
        model, full_train_loader, target_epochs=best_epoch
    )
    print(f"    Train time: {train_time:.2f}s")

    # === ОЦЕНКА на test ===
    history_df = pd.concat([sneakers_train_norm, sneakers_val_norm])
    predictions, inference_time, mae, rmse, mape = evaluate_mlp(
        model,
        history_df,
        sneakers_test_norm,
        feature_cols,
        target_col,
        sneakers_scaler,
        sneakers_scaled_cols,
    )

    print(
        f"    MAE: {mae:.4f}, RMSE: {rmse:.4f}, MAPE: {mape:.2f}%, Inference: {inference_time:.4f}s"
    )

    all_mlp_results.append(
        {
            "ticker": ticker,
            "model": "MLP",
            "best_epoch": best_epoch,
            "train_time": train_time,
            "inference_time": inference_time,
            "mae": mae,
            "rmse": rmse,
            "mape": mape,
            "best_val_loss": best_val_loss,
        }
    )

# Сводная таблица
mlp_results_df = pd.DataFrame(all_mlp_results)
print("\n" + "=" * 70)
print(mlp_results_df.to_string(index=False))

 Автомобили
 Volkswagen
    Признаков: 36
    Первый проход (определение best_epoch)...
        Epoch 1: train_loss=0.188116, val_loss=0.102223
        Epoch 2: train_loss=0.123070, val_loss=0.083228
        Epoch 3: train_loss=0.104354, val_loss=0.048629
        Epoch 4: train_loss=0.084576, val_loss=0.043553
        Epoch 5: train_loss=0.064218, val_loss=0.034858
    Best epoch: 69, val loss: 0.002254
    Второй проход (обучение на train+val, 69 эпох)...
    Train time: 14.88s
    MAE: 13.7730, RMSE: 14.7548, MAPE: 96.25%, Inference: 2.7675s
 Кроссовки
 Nike
    Признаков: 36
    Первый проход (определение best_epoch)...
        Epoch 1: train_loss=0.079736, val_loss=0.006139
        Epoch 2: train_loss=0.018621, val_loss=0.006823
        Epoch 3: train_loss=0.014380, val_loss=0.005485
        Epoch 4: train_loss=0.013952, val_loss=0.004351
        Epoch 5: train_loss=0.010236, val_loss=0.007163
    Best epoch: 9, val loss: 0.002221
    Второй проход (обучение на train+val, 9 эпох)..

In [ ]:
mlp_results_df.to_csv("./results/mlp_results.csv", index=False)

# Gradient Boosting